# QLoRA: LoRA on a 4-bit base model

QLoRA (Dettmers et al., 2023) trains the same `B A` adapters as LoRA, but stores the frozen `W` in **4 bits** instead of 16:

$$y = x\, \text{dequant}(W_{4bit})^\top + \frac{\alpha}{r}\, x A^\top B^\top$$

- The frozen weights take about a quarter of the memory. For a 70B model that is 140 GB → 35 GB.
- Gradients still flow through the dequantized `W` to reach earlier adapters. `W` itself never changes, so it never needs full precision storage.
- The adapters stay in 16/32 bits. They are small, so this costs almost nothing.

This notebook builds the 4-bit format by hand in PyTorch, with no `bitsandbytes`:

- **NF4 (4-bit NormalFloat).** Pretrained weights are roughly normally distributed. NF4 places its 16 levels on quantiles of a normal distribution, so each level is used about equally often.
- **Block-wise absmax.** Each block of 64 weights is divided by its largest absolute value, which maps it into [-1, 1]. Then every weight is snapped to the nearest NF4 level. The 4-bit index and one scale per block are stored.
- **Dequantize on the fly.** Each forward (and backward) rebuilds the bf16 matrix from the indices, uses it, and throws it away.

The paper also uses double quantization (the block scales are quantized too) and paged optimizers. They are left out here to keep the code short.

## NF4 quantization

The 16 NF4 levels from the QLoRA paper. `torch.bucketize` against the midpoints finds the nearest level. Two 4-bit indices are packed into one byte.

In [ ]:
import torch

NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0,
])


def quantize_nf4(weight: torch.Tensor, block_size: int = 64) -> tuple[torch.Tensor, torch.Tensor]:
    """Return packed 4-bit indices (uint8, two per byte) and one absmax per block."""
    if weight.numel() % block_size:
        raise ValueError("weight size must be a multiple of block_size")
    blocks = weight.float().reshape(-1, block_size)
    absmax = blocks.abs().amax(dim=1, keepdim=True).clamp_min(1e-12)
    levels = NF4_LEVELS.to(weight.device)
    midpoints = (levels[1:] + levels[:-1]) / 2
    indices = torch.bucketize(blocks / absmax, midpoints).to(torch.uint8).flatten()
    packed = (indices[0::2] << 4) | indices[1::2]
    return packed, absmax.squeeze(1)


def dequantize_nf4(packed, absmax, shape, levels, dtype=torch.bfloat16) -> torch.Tensor:
    """Rebuild the weight from packed indices and block scales."""
    indices = torch.stack((packed >> 4, packed & 0x0F), dim=1).flatten().long()
    blocks = levels[indices].reshape(len(absmax), -1) * absmax[:, None]
    return blocks.reshape(shape).to(dtype)


# The toy matrix from the diagram: 800 x 500, LoRA rank 4.
d_out, d_in, r = 800, 500, 4
W = torch.randn(d_out, d_in) * 0.02
packed, absmax = quantize_nf4(W)
W_back = dequantize_nf4(packed, absmax, W.shape, NF4_LEVELS, dtype=torch.float32)

print(f"16-bit: {W.numel() * 2:,} bytes")
print(f"4-bit:  {packed.numel():,} bytes of indices + {absmax.numel() * 4:,} bytes of scales")
print(f"relative error: {((W_back - W).norm() / W.norm()).item():.3f}")
print(f"trainable LoRA parameters on top: {d_out * r + r * d_in:,} (same as LoRA)")

## Setup

Same base model and tokenizer as `finetuning.ipynb`. Run `shop_assistant_dataset.ipynb` first so `data/instruction-data.json` exists.

In [ ]:
import json
import sys
import time
from functools import partial
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from reasoning_from_scratch.qwen3 import (
    QWEN_CONFIG_06_B,
    KVCache,
    Qwen3Model,
    Qwen3Tokenizer,
    download_qwen3_small,
)

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("No CUDA GPU found. Everything runs on CPU, but training will be slow.")

download_qwen3_small(kind="base", out_dir=ROOT / "qwen3")
tokenizer = Qwen3Tokenizer(ROOT / "qwen3" / "tokenizer-base.json")
model = Qwen3Model(QWEN_CONFIG_06_B)
weights = torch.load(ROOT / "qwen3" / "qwen3-0.6B-base.pth", map_location="cpu", weights_only=True)
model.load_state_dict(weights)
del weights
model.to(device)
model.eval()

n_base = sum(param.numel() for param in model.parameters())
print(f"base model: {n_base / 1e6:.1f}M parameters on {device}")

In [ ]:
INSTRUCTION = (
    "You are a shop assistant for a clothing and shoe store. "
    "Read the customer message and return only the next action as JSON "
    "with keys action, slots, missing, and reply. "
    "Do not invent products, prices, or stock."
)


def format_input(entry: dict) -> str:
    """Format an instruction entry as the training prompt."""
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text


class InstructionDataset(Dataset):
    """Prompt plus response, with the prompt length kept for the loss mask."""

    def __init__(self, data: list[dict], tokenizer) -> None:
        self.examples = []
        for entry in data:
            prompt = format_input(entry) + "\n\n### Response:\n"
            full = prompt + entry["output"]
            prompt_ids = tokenizer.encode(prompt)
            full_ids = tokenizer.encode(full)
            self.examples.append((full_ids, len(prompt_ids)))

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.examples[index]

    def __len__(self) -> int:
        return len(self.examples)


def collate(batch, device, pad_id: int):
    """Pad a batch and ignore prompt and padding tokens in the loss."""
    sequences, prompt_lens = [], []
    for ids, prompt_len in batch:
        sequences.append(ids + [pad_id])
        prompt_lens.append(prompt_len)
    max_len = max(len(seq) for seq in sequences)
    input_rows, target_rows = [], []
    for seq, prompt_len in zip(sequences, prompt_lens):
        padded = seq + [pad_id] * (max_len - len(seq))
        inputs = torch.tensor(padded[:-1], device=device)
        targets = torch.tensor(padded[1:], device=device)
        targets[: prompt_len - 1] = -100
        pad_positions = (targets == pad_id).nonzero(as_tuple=True)[0]
        if len(pad_positions) > 1:
            targets[pad_positions[1:]] = -100
        input_rows.append(inputs)
        target_rows.append(targets)
    return torch.stack(input_rows), torch.stack(target_rows)


with open(ROOT / "data" / "instruction-data.json", encoding="utf-8") as f:
    data = json.load(f)

train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion :]

collate_fn = partial(collate, device=device, pad_id=tokenizer.pad_token_id)
train_loader = DataLoader(
    InstructionDataset(train_data, tokenizer),
    batch_size=1,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
val_loader = DataLoader(
    InstructionDataset(val_data, tokenizer),
    batch_size=1,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=0,
)
print("train", len(train_data), "val", len(val_data), "test", len(test_data))

In [ ]:
# Every linear layer inside the 28 transformer blocks gets an adapter.
# The token embedding, the norms, and the output head stay as they are.
TARGETS = {"att": ("W_query", "W_key", "W_value", "out_proj"), "ff": ("fc1", "fc2", "fc3")}


def target_layers(model):
    """Yield (parent module, attribute name) for every adapted linear layer."""
    for block in model.trf_blocks:
        for parent_name, names in TARGETS.items():
            parent = getattr(block, parent_name)
            for name in names:
                yield parent, name


def add_adapters(model, wrap) -> None:
    """Freeze every base weight, then replace each target layer with wrap(layer)."""
    for param in model.parameters():
        param.requires_grad = False
    for parent, name in target_layers(model):
        setattr(parent, name, wrap(getattr(parent, name)))


def count_trainable(model) -> int:
    """Count trainable parameters. Shared parameters are counted once."""
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


@torch.no_grad()
def evaluate_loss(model, loader) -> float:
    """Mean masked cross-entropy over a loader."""
    model.eval()
    losses = []
    for inputs, targets in loader:
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
        losses.append(loss.item())
    return sum(losses) / len(losses)


def train(model, params, lr: float, accum_steps: int = 4, weight_decay: float = 0.0, log_every: int = 40) -> None:
    """One pass over the training split. Gradients from 4 examples make one AdamW step."""
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    optimizer.zero_grad()
    start = time.time()
    for step, (inputs, targets) in enumerate(train_loader, start=1):
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
        (loss / accum_steps).backward()
        if step % accum_steps == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
        if step % log_every == 0 or step == len(train_loader):
            print(f"step {step}/{len(train_loader)} loss {loss.item():.3f}")
    model.eval()
    print(f"training took {time.time() - start:.0f}s")
    if device.type == "cuda":
        print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")


def prompt_for(customer: str) -> str:
    """Wrap a customer message in the training prompt."""
    return format_input({"instruction": INSTRUCTION, "input": customer}) + "\n\n### Response:\n"


def generate(model, prompt: str, max_new_tokens: int = 100) -> str:
    """Greedy decoding with a KV cache. Stops at <|endoftext|>."""
    token_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    new_ids = []
    with torch.inference_mode():
        logits = model(token_ids, cache=cache)
        for _ in range(max_new_tokens):
            next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
            if next_id.item() == tokenizer.eos_token_id:
                break
            new_ids.append(next_id.item())
            logits = model(next_id, cache=cache)
    model.reset_kv_cache()
    return tokenizer.decode(new_ids).strip()


def score_actions(model, rows: list[dict]) -> None:
    """Generate an action for each row and compare it with the expected one."""
    valid = correct = 0
    for entry in rows:
        text = generate(model, prompt_for(entry["input"]))
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, dict):
            continue
        valid += 1
        correct += payload.get("action") == json.loads(entry["output"])["action"]
    print(f"valid JSON {valid}/{len(rows)}   right action {correct}/{len(rows)}")


SAMPLE = "Busco tenis para correr, talla 42, que no sean negros"

## A 4-bit linear layer

`NF4Linear` stores only the packed indices and scales as buffers. `bfloat16` weights exist only while one layer runs.

A plain `x @ W.T` would make autograd keep the dequantized `W` of every layer until the backward pass, which defeats the purpose. The custom autograd function keeps nothing and dequantizes again in the backward pass. No gradient is computed for `W`, only for `x`, so the adapters in earlier layers still learn.

In [ ]:
class _NF4Matmul(torch.autograd.Function):
    """x @ W.T where W is dequantized in forward and again in backward."""

    @staticmethod
    def forward(ctx, x, layer):
        ctx.layer = layer
        return x @ layer.dequantize().T

    @staticmethod
    def backward(ctx, grad_out):
        return grad_out @ ctx.layer.dequantize(), None


class NF4Linear(nn.Module):
    """Frozen linear layer stored in 4-bit NormalFloat."""

    def __init__(self, linear: nn.Linear, block_size: int = 64) -> None:
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.dtype = linear.weight.dtype
        packed, absmax = quantize_nf4(linear.weight.data, block_size)
        self.register_buffer("packed", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("levels", NF4_LEVELS.to(packed.device))

    def dequantize(self) -> torch.Tensor:
        return dequantize_nf4(
            self.packed, self.absmax, (self.out_features, self.in_features), self.levels, self.dtype
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return _NF4Matmul.apply(x, self)


def layer_bytes(model) -> int:
    """Bytes held by the target linear layers (weights or 4-bit buffers)."""
    total = 0
    for parent, name in target_layers(model):
        layer = getattr(parent, name)
        total += sum(t.numel() * t.element_size() for t in [*layer.parameters(), *layer.buffers()])
    return total

## Quantize the base model

First the bf16 validation loss, then quantize the 196 linear layers in the blocks, then the same loss again. The gap is what 4 bits cost before any training. The embedding and the output head stay in bf16, as in the paper.

In [ ]:
val_bf16 = evaluate_loss(model, val_loader)
bytes_bf16 = layer_bytes(model)
reference = model.trf_blocks[0].ff.fc1.weight.detach().clone()


def quantize_model(model) -> None:
    """Replace every target linear layer with a frozen NF4Linear."""
    for param in model.parameters():
        param.requires_grad = False
    for parent, name in target_layers(model):
        setattr(parent, name, NF4Linear(getattr(parent, name)))
    if device.type == "cuda":
        torch.cuda.empty_cache()


quantize_model(model)
bytes_nf4 = layer_bytes(model)
restored = model.trf_blocks[0].ff.fc1.dequantize()
val_nf4 = evaluate_loss(model, val_loader)

print(f"block linear layers: {bytes_bf16 / 1024**3:.2f} GiB (bf16) -> {bytes_nf4 / 1024**3:.2f} GiB (NF4)")
print(f"relative error on block 1 fc1: {((restored.float() - reference.float()).norm() / reference.float().norm()).item():.3f}")
print(f"validation loss: bf16 {val_bf16:.3f}   NF4 {val_nf4:.3f}")
if device.type == "cuda":
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

## LoRA on top

The same `LoRALinear` as in `lora.ipynb`, wrapped around each `NF4Linear`. Rank 8, `α = 16`.

In [ ]:
class LoRALinear(nn.Module):
    """Frozen (here 4-bit) layer plus a trainable low-rank update (alpha / r) * B @ A."""

    def __init__(self, base: nn.Module, rank: int, alpha: float) -> None:
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(rank, base.in_features, device=device))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank, device=device))
        nn.init.kaiming_uniform_(self.A, a=5**0.5)
        self.scaling = alpha / rank

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        update = (x.float() @ self.A.T) @ self.B.T
        return self.base(x) + (update * self.scaling).to(x.dtype)


RANK, ALPHA = 8, 16
add_adapters(model, lambda layer: LoRALinear(layer, RANK, ALPHA))

n_trainable = count_trainable(model)
print(f"trainable {n_trainable:,} / {n_base:,} base parameters ({100 * n_trainable / n_base:.2f}%)")

In [ ]:
val_before = val_nf4
print("4-bit base model output:")
print(generate(model, prompt_for(SAMPLE)))

## Training

Same loop as `finetuning.ipynb`: one example at a time, the loss only scores the action, and 4 gradients add up before each AdamW step. Only the adapter parameters go to the optimizer.

In [ ]:
qlora_params = [param for param in model.parameters() if param.requires_grad]
train(model, qlora_params, lr=2e-4)

In [ ]:
val_after = evaluate_loss(model, val_loader)
print(f"validation loss: {val_before:.3f} -> {val_after:.3f}")
print("finetuned output:")
print(generate(model, prompt_for(SAMPLE)))
score_actions(model, test_data)

## Save the adapter

The adapters were trained against the **4-bit** weights. They are meant to run on top of the same quantized base. If you merge them into the bf16 weights instead, the result differs slightly from what was trained. That is why QLoRA adapters are usually kept separate, or the merged model is quantized again.

In [ ]:
adapter = {name: param.detach().cpu() for name, param in model.named_parameters() if param.requires_grad}
adapter_path = ROOT / "qwen3" / "shop-qlora-adapter.pth"
torch.save(adapter, adapter_path)
print(f"adapter: {len(adapter)} tensors, {adapter_path.stat().st_size / 1024**2:.1f} MiB")

## Recap

- Same trainable parameters as LoRA. What changes is the memory for the **frozen** weights: about 4.5 bits per weight here (4 bits plus a float32 scale per 64 weights).
- Each step dequantizes every layer twice (forward and backward), so it trades speed for memory. `bitsandbytes` does the same with fused CUDA kernels.
- On a 0.6B model the savings are small in GiB. On 7B–70B models, this is what makes finetuning fit on a single GPU.